# Run the Car Price Predictor Streamlit app


In [ ]:
!pip install streamlit catboost scikit-learn joblib pandas pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 105.3 MB/s eta 0:00:00


In [5]:
%%writefile /content/app.py
import json
from pathlib import Path

import joblib
import pandas as pd
import streamlit as st

st.set_page_config(
    page_title="Car Price Predictor",
    layout="centered",
)

try:
    ARTIFACT_DIR = Path(__file__).parent
except NameError:
    ARTIFACT_DIR = Path.cwd()

@st.cache_resource
def load_artifacts():
    preprocessor = joblib.load(ARTIFACT_DIR / "preprocessor.pkl")
    cat_model = joblib.load(ARTIFACT_DIR / "cat_model.pkl")
    with open(ARTIFACT_DIR / "options.json") as f:
        options = json.load(f)
    return preprocessor, cat_model, options


try:
    preprocessor, cat_model, options = load_artifacts()
except FileNotFoundError:
    st.error(
        "Missing model files. Make sure `preprocessor.pkl`, `cat_model.pkl`, "
        "and `options.json` are in the same folder as `app.py`. "
        "Run the `save_artifacts_cell.py` cell in your Colab notebook to "
        "generate them, then download them here."
    )
    st.stop()

st.title("Used Car Price Predictor")
st.write(
    "Fill in the car's details and get an estimated resale price, "
    "predicted by a CatBoost model trained on used-car listings."
)

st.divider()

with st.form("car_form"):
    col1, col2 = st.columns(2)

    with col1:
        company = st.selectbox("Company", options["companies"])

        models_for_company = options["models_by_company"].get(company, [])
        model_name = st.selectbox("Model", models_for_company)

        year = st.number_input(
            "Year",
            min_value=options["year_min"],
            max_value=2026,
            value=min(2018, 2026),
            step=1,
        )

        km_driven = st.number_input(
            "KM Driven",
            min_value=0,
            max_value=int(options["km_driven_max"]) * 2,
            value=50000,
            step=1000,
        )

    with col2:
        fuel = st.selectbox("Fuel Type", options["fuel"])
        seller_type = st.selectbox("Seller Type", options["seller_type"])
        transmission = st.selectbox("Transmission", options["transmission"])
        owner = st.selectbox("Owner", options["owner"])

    submitted = st.form_submit_button("Predict Price", use_container_width=True)

if submitted:
    input_df = pd.DataFrame(
        [
            {
                "year": year,
                "km_driven": km_driven,
                "fuel": fuel,
                "seller_type": seller_type,
                "transmission": transmission,
                "owner": owner,
                "company": company,
                "model": model_name,
            }
        ]
    )

    try:
        X_transformed = preprocessor.transform(input_df)
        prediction = cat_model.predict(X_transformed)[0]
        prediction = max(0, prediction)  # prices can't be negative

        st.divider()
        st.metric("Estimated Selling Price", f"${prediction:,.0f}")
        st.caption(
            f"{year} {company} {model_name} · {km_driven:,} km · "
            f"{fuel} · {transmission} · {owner}"
        )
    except Exception as e:
        st.error(f"Something went wrong while predicting: {e}")

st.divider()
st.caption(
    "Model: CatBoostRegressor · Prices in USD, converted from the original "
    "INR dataset."
)


Overwriting /content/app.py


In [6]:
get_ipython().system_raw('streamlit run /content/app.py --server.port 8501 &>/content/logs.txt &')
import time; time.sleep(5)  # give the server a moment to start
print("Server starting — check logs.txt if the tunnel below doesn't work")

Server starting — check logs.txt if the tunnel below doesn't work


In [7]:
from pyngrok import ngrok

NGROK_AUTHTOKEN = "3IGg3fD3lZCZFIab93luTKgK7kD_5wFm9XhbNwoh4p1hQLXBU"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Kill any existing tunnels first (avoids 'tunnel already exists' errors on re-run)
ngrok.kill()

public_url = ngrok.connect(8501, "http")
print("Your app is live at:", public_url)

Your app is live at: NgrokTunnel: "https://rewash-undocked-karaoke.ngrok-free.dev" -> "http://localhost:8501"
